In [1]:
import pandas as pd
df = pd.read_csv("titanic_data.csv")

In [6]:
print("Shape:", df.shape)
display(df.head(5))

print("\n--- INFO ---")
print(df.info())

print("\n--- DTYPES ---")
print(df.dtypes)

print("\n--- Missing values ---")
print(df.isnull().sum().sort_values(ascending=False))

print("\n--- Duplicates ---")
print("Duplicate rows:", df.duplicated().sum())

Shape: (782, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True



--- INFO ---
<class 'pandas.core.frame.DataFrame'>
Index: 782 entries, 0 to 888
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     782 non-null    int64  
 1   pclass       782 non-null    int64  
 2   sex          782 non-null    object 
 3   age          677 non-null    float64
 4   sibsp        782 non-null    int64  
 5   parch        782 non-null    int64  
 6   fare         782 non-null    float64
 7   embarked     780 non-null    object 
 8   class        782 non-null    object 
 9   who          782 non-null    object 
 10  adult_male   782 non-null    bool   
 11  deck         202 non-null    object 
 12  embark_town  780 non-null    object 
 13  alive        782 non-null    object 
 14  alone        782 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 87.1+ KB
None

--- DTYPES ---
survived         int64
pclass           int64
sex             object
age       

In [3]:
#Titanic’te sık görülenler: Sex, Embarked, Cabin, Ticket gibi sütunlar kategorik, Survived hedef (varsa), Age, Fare sayısal.
# Yaygın titanic kolonları için güvenli tip ayarı (kolon varsa uygular)
categorical_candidates = ["Sex", "Embarked", "Cabin", "Ticket", "Name"]
for col in categorical_candidates:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Sayısal olması gereken ama object gelenleri dönüştür
numeric_candidates = ["Age", "Fare", "SibSp", "Parch", "Pclass"]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [4]:
# Duplikasyon temizliği
df = df.drop_duplicates()

# Opsiyonel: ID kolonlarını düşür (varsa)
drop_cols = []
for col in ["PassengerId", "Name", "Ticket", "Cabin"]:
    if col in df.columns:
        drop_cols.append(col)

print("Dropping cols:", drop_cols)
df_clean = df.drop(columns=drop_cols) if drop_cols else df.copy()

print(df_clean.shape)

Dropping cols: []
(782, 15)


In [ ]:
#Aykırı değer
def iqr_outlier_summary(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    out_count = ((df[col] < low) | (df[col] > high)).sum()
    return {"Q1": q1, "Q3": q3, "IQR": iqr, "low": low, "high": high, "outliers": int(out_count)}

for col in ["Age", "Fare"]:
    if col in df_clean.columns:
        print(col, iqr_outlier_summary(df_clean, col))

In [ ]:
from sklearn.model_selection import train_test_split

target = "Survived"  # Titanic standart hedef
if target not in df_clean.columns:
    raise ValueError(f"'{target}' sütunu bulunamadı. Kolonlar: {list(df_clean.columns)}")

X = df_clean.drop(columns=[target])
y = df_clean[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# X_train / X_test dönüştür
X_train_prep = preprocess.fit_transform(X_train)  # fit sadece train
X_test_prep  = preprocess.transform(X_test)

print("After preprocess -> Train:", X_train_prep.shape, "Test:", X_test_prep.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))